# Section 2 · Notebook 2 — Prebuilt Tools and AWS Integration (Groq edition)

> **Groq edition.** Same lesson as the Bedrock notebook next to this one; only the model setup changes. Needs a `GROQ_API_KEY` (free at [console.groq.com/keys](https://console.groq.com/keys)) in your environment or a `.env` file. Parts 1–2 need nothing else. Part 3 drives AWS itself (S3, DynamoDB, Lambda), so it still needs AWS credentials — but no Bedrock access.

1. The prebuilt calculator tool
2. Combining several prebuilt tools in one request
3. Talking to AWS in plain English with `use_aws` — S3, DynamoDB and Lambda

> Part 3 creates a few small AWS resources (an S3 bucket, a DynamoDB table, a Lambda function and its IAM role). The last cell deletes all of them.

In [ ]:
%pip install -q -r requirements-groq.txt

In [ ]:
import os

from dotenv import load_dotenv
from strands.models.openai import OpenAIModel

load_dotenv()  # reads GROQ_API_KEY from a .env file, if present

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")
# Part 3 still talks to AWS (S3, DynamoDB, Lambda), so it needs a Region and AWS credentials.
REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
os.environ.setdefault("AWS_REGION", REGION)

model = OpenAIModel(
    client_args={
        "api_key": os.environ["GROQ_API_KEY"],
        "base_url": os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),  # or a local OpenAI-compatible server
        "max_retries": 5,  # retry dropped connections (the client default is 2)
    },
    model_id=MODEL_ID,
    # Groq reasoning models stream their thinking by default; hide it so only the answer prints.
    params={"extra_body": {"include_reasoning": False}},
)
print(f"Using {MODEL_ID} on Groq (AWS Region for part 3: {REGION})")

---
## 1. Calculator Tool

Expressions, equations and calculus — all from a single import.

In [ ]:
from strands import Agent
from strands_tools import calculator

agent = Agent(model=model, tools=[calculator], system_prompt="You are a helpful math assistant.")

agent("What's 42 raised to the power of 9?")
agent("Solve the equation x^2 + 5x + 6 = 0")
agent("What's the derivative of sin(x) * cos(x)?")

---
## 2. Combining Multiple Tools

Fetch live data, crunch the numbers, and save the result — one request, three tools.

`file_write` normally asks you to confirm before writing a file. `BYPASS_TOOL_CONSENT=true` skips that prompt for this demo; leave it off in real applications.

In [ ]:
from strands_tools import file_write, http_request

os.environ["BYPASS_TOOL_CONSENT"] = "true"

agent = Agent(
    model=model,
    tools=[http_request, calculator, file_write],
    system_prompt="You help with data analysis tasks.",
)

agent("""
Fetch the 7-day forecast from
https://api.open-meteo.com/v1/forecast?latitude=12.97&longitude=77.59&daily=temperature_2m_max,temperature_2m_min&timezone=Asia/Kolkata
extract the daily maximum temperatures for Bangalore,
calculate the average maximum temperature over the period,
and save a short summary to weather_summary.txt
""")

In [ ]:
if os.path.exists("weather_summary.txt"):
    print(open("weather_summary.txt").read())
else:
    print("No weather_summary.txt yet — scroll up to see which tools the agent called.")

---
## 3. AWS Integration with `use_aws`

`use_aws` turns plain English into AWS API calls. It uses the same credentials as your notebook, so it can only do what your IAM permissions allow.

### 3.0 Create the demo resources

This cell creates:
- an S3 bucket with a `sales-2024.csv` file
- a DynamoDB table `customers-<suffix>` with customer `12345`
- a Lambda function `order-processor-<suffix>` (plus an IAM role for it)

A random suffix keeps the names unique, so nothing you already have is touched.

In [ ]:
import csv
import io
import json
import time
import zipfile

import boto3

session = boto3.Session(region_name=REGION)
account_id = session.client("sts").get_caller_identity()["Account"]
s3 = session.client("s3")
dynamodb = session.client("dynamodb")
iam = session.client("iam")
lambda_client = session.client("lambda")

import uuid

suffix = uuid.uuid4().hex[:6]  # unique names, so we never touch resources you already have
BUCKET = f"agent-tools-demo-{account_id}-{REGION}-{suffix}"
TABLE = f"customers-{suffix}"
FUNCTION = f"order-processor-{suffix}"
ROLE_NAME = f"order-processor-demo-role-{suffix}"

# --- S3 bucket + CSV -------------------------------------------------------
if REGION == "us-east-1":
    s3.create_bucket(Bucket=BUCKET)
else:
    s3.create_bucket(Bucket=BUCKET, CreateBucketConfiguration={"LocationConstraint": REGION})

rows = [
    ("month", "region", "revenue"),
    ("2024-01", "North", 120000), ("2024-02", "North", 135000), ("2024-03", "North", 142000),
    ("2024-01", "South", 98000), ("2024-02", "South", 104000), ("2024-03", "South", 111000),
]
buf = io.StringIO()
csv.writer(buf).writerows(rows)
s3.put_object(Bucket=BUCKET, Key="sales-2024.csv", Body=buf.getvalue().encode())
print(f"S3 bucket ready: s3://{BUCKET}/sales-2024.csv")

# --- DynamoDB table + one customer -----------------------------------------
dynamodb.create_table(
    TableName=TABLE,
    KeySchema=[{"AttributeName": "customer_id", "KeyType": "HASH"}],
    AttributeDefinitions=[{"AttributeName": "customer_id", "AttributeType": "S"}],
    BillingMode="PAY_PER_REQUEST",
)
dynamodb.get_waiter("table_exists").wait(TableName=TABLE)
dynamodb.put_item(
    TableName=TABLE,
    Item={"customer_id": {"S": "12345"}, "name": {"S": "Asha Rao"}, "email": {"S": "asha@example.com"}},
)
print(f"DynamoDB table ready: {TABLE}")

# --- Lambda function ---------------------------------------------------------
trust = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"}, "Action": "sts:AssumeRole"}],
}
role_arn = iam.create_role(RoleName=ROLE_NAME, AssumeRolePolicyDocument=json.dumps(trust))["Role"]["Arn"]
iam.attach_role_policy(RoleName=ROLE_NAME, PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
time.sleep(10)  # let IAM propagate

handler_code = '''
import json

def handler(event, context):
    order_id = event.get("order_id", "unknown")
    return {"order_id": order_id, "status": "PROCESSED", "message": f"Order {order_id} processed successfully"}
'''
zip_buf = io.BytesIO()
with zipfile.ZipFile(zip_buf, "w") as z:
    z.writestr("lambda_function.py", handler_code)

lambda_client.create_function(
    FunctionName=FUNCTION,
    Runtime="python3.12",
    Role=role_arn,
    Handler="lambda_function.handler",
    Code={"ZipFile": zip_buf.getvalue()},
    Timeout=10,
)
lambda_client.get_waiter("function_active_v2").wait(FunctionName=FUNCTION)
print(f"Lambda function ready: {FUNCTION}")

### 3.1 List your S3 buckets

> **Groq note:** `use_aws` lets the model pick an AWS profile with `profile_name`, and `gpt-oss` models like to send `profile_name="default"`, which overrides an `AWS_PROFILE` you set (an SSO profile, for example). So we hand the agent a copy of the tool with that option removed: every call uses this notebook's credentials. It is the same idea as Section 3, where the memory tool's user is fixed in code — decide in code what the model must not choose.

In [ ]:
import copy

from strands.tools.tools import PythonAgentTool
from strands_tools import use_aws as use_aws_module

# use_aws without the profile_name option (see the note above)
spec = copy.deepcopy(use_aws_module.TOOL_SPEC)
del spec["inputSchema"]["json"]["properties"]["profile_name"]
use_aws = PythonAgentTool("use_aws", spec, use_aws_module.use_aws)

aws_agent = Agent(
    model=model,
    tools=[use_aws, calculator],
    system_prompt=f"You are an AWS assistant that helps manage cloud resources. Use region {REGION}.",
)

aws_agent("List all S3 buckets in my account")

### 3.2 Data analysis on S3

Read a CSV from S3, total the revenue, and write a summary back to S3.

In [ ]:
# Fresh conversation per task: Groq's free tier rejects any single request over 8,000 tokens
aws_agent.messages.clear()

aws_agent(
    f"Read the sales data from S3 bucket '{BUCKET}' key 'sales-2024.csv', "
    "calculate the total revenue, and save a short summary back to the same bucket as 'sales-summary.txt'"
)

### 3.3 Customer service on DynamoDB

In [ ]:
aws_agent.messages.clear()  # fresh conversation per task (see 3.2)

aws_agent(
    f"Look up customer ID 12345 in the DynamoDB table '{TABLE}' and update their email to newemail@example.com. "
    "Then read the item back to confirm."
)

### 3.4 Workflow orchestration with Lambda

In [ ]:
aws_agent.messages.clear()  # fresh conversation per task (see 3.2)

aws_agent(f"Invoke the Lambda function '{FUNCTION}' with the payload {{\"order_id\": \"67890\"}} and tell me the result")

### 3.5 Clean up

Deletes everything created in 3.0.

In [ ]:
# S3: delete all objects, then the bucket
for obj in s3.list_objects_v2(Bucket=BUCKET).get("Contents", []):
    s3.delete_object(Bucket=BUCKET, Key=obj["Key"])
s3.delete_bucket(Bucket=BUCKET)
print("Deleted bucket", BUCKET)

dynamodb.delete_table(TableName=TABLE)
print("Deleted table", TABLE)

lambda_client.delete_function(FunctionName=FUNCTION)
iam.detach_role_policy(RoleName=ROLE_NAME, PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
iam.delete_role(RoleName=ROLE_NAME)
print("Deleted Lambda function and role")